# Film Diary ML: Train/Test Prototype

This notebook loads the app-exported ML dataset, creates an explicit random train/test split stratified by release decade, trains the standard-library logistic regression baselines, evaluates them against the V1 satisfaction score baseline, and writes app-consumable prediction artifacts.

In [ ]:
from pathlib import Path
import json
import sys

cwd = Path.cwd()
repo_root = next((candidate for candidate in [cwd, *cwd.parents] if (candidate / 'ml' / 'film_ml').exists()), cwd)
sys.path.insert(0, str(repo_root / 'ml'))

dataset_path = repo_root / 'local' / 'film-diary-ml-dataset-v1.json'
output_dir = repo_root / 'ml' / 'outputs'
test_fraction = 0.20
random_state = 42
min_class_count = 12
ks = [5, 10, 25]

print(repo_root)
print(dataset_path)

In [ ]:
from film_ml import TrainingConfig, load_dataset, train_and_evaluate, watched_rows, write_training_artifacts
from film_ml.dataset import release_decade, split_train_test_by_decade

dataset = load_dataset(dataset_path)
rows = dataset['rows']
watched = watched_rows(rows)
train_rows, test_rows, split_summary = split_train_test_by_decade(
    watched,
    test_fraction=test_fraction,
    random_state=random_state,
)

print(json.dumps({
    'total_rows': len(rows),
    'watched_rows': len(watched),
    'candidate_rows': len([row for row in rows if row.get('rowKind') == 'candidate']),
    'split': split_summary,
}, indent=2))

In [ ]:
decade_table = []
for decade, counts in split_summary['groups'].items():
    decade_table.append({
        'decade': decade,
        'total': counts['total'],
        'train': counts['train'],
        'test': counts['test'],
    })

decade_table

In [ ]:
config = TrainingConfig(
    test_fraction=test_fraction,
    min_class_count=min_class_count,
    ks=ks,
    random_state=random_state,
)
run = train_and_evaluate(dataset, config, dataset_path=dataset_path)
write_training_artifacts(run, output_dir, source_dataset=dataset_path)

print(json.dumps(run.metrics, indent=2))

In [ ]:
def top_weights(model, limit=20):
    weights = model.get('weights', {})
    return sorted(weights.items(), key=lambda item: abs(item[1]), reverse=True)[:limit]

for target, model in run.models.items():
    print(f'\n{target}')
    for key, weight in top_weights(model):
        print(f'{weight: .4f}  {key}')

In [ ]:
candidate_predictions = [
    prediction
    for prediction in run.predictions
    if prediction.get('rowKind') == 'candidate'
    and isinstance(prediction.get('mlHighRatingProbability'), float)
]

sorted(
    candidate_predictions,
    key=lambda prediction: prediction['mlHighRatingProbability'],
    reverse=True,
)[:20]